In [ ]:
from utils import load_experiment_results
import numpy as np
from scipy.stats import norm as normal_dist
import json

experiment_name = 'model_family_comparison'
datasets = ['glassdoor']
results_sets = [load_experiment_results(dataset, experiment_name) for dataset in datasets]

# Load the glassdoor variables to get population stds
vars_file = json.load(open('../data/variables/glassdoor_variables.json', 'r'))

# Let's check the log prob calculation for a few examples
print("Debugging log probability calculations for Glassdoor:\n")
print("=" * 80)

results = results_sets[0]
# Filter to just normal distributions and a few examples
normal_results = results[results['ground_truth_distribution_type'] == 'normal'].head(10)

for idx, row in normal_results.iterrows():
    var_name = row['variable']
    var_info = vars_file.get(var_name, {})
    pop_std = var_info.get('std', None)
    
    if pop_std is None:
        continue
    
    # Recalculate log prob components
    log_prob_under_prior = normal_dist.logpdf(row['ground_truth'], loc=row['mean'], scale=row['std'])
    jeffreys_adjustment = np.log(pop_std)
    stored_log_prob = row['ground_truth_log_prob']
    
    print(f"\nVariable: {var_name}")
    print(f"  Mean: {row['mean']:.2f}, Std: {row['std']:.2f}, Ground Truth: {row['ground_truth']:.2f}")
    print(f"  Log prob under prior: {log_prob_under_prior:.6f}")
    print(f"  Population std: {pop_std:.2f}")
    print(f"  Jeffreys adjustment: {jeffreys_adjustment:.6f}")
    print(f"  Stored log prob (with Jeffreys): {stored_log_prob:.6f}")
    print(f"  Difference: {log_prob_under_prior - stored_log_prob:.6f} (should match Jeffreys)")
    
print("\n" + "=" * 80)
print("\nCONCLUSION: The Jeffreys prior adjustment is being SUBTRACTED from the log prob,")
print("making all log probs appear much lower (more negative) than they actually are.")
print("\nFor Glassdoor with large salary values (std ~30K), the adjustment is ~10,")
print("which makes a log prob of -9.65 appear as -19.93!")

In [ ]:
# Let's compare the Jeffreys adjustments across different datasets
import pandas as pd

datasets_to_check = ['glassdoor', 'nhanes', 'census']
adjustment_summary = []

for dataset in datasets_to_check:
    try:
        vars_file = json.load(open(f'../data/variables/{dataset}_variables.json', 'r'))
        
        adjustments = []
        for var_name, var_info in vars_file.items():
            if 'std' in var_info and var_info.get('ground_truth_distribution_type') == 'normal':
                jeffreys_adj = np.log(var_info['std'])
                adjustments.append(jeffreys_adj)
        
        if adjustments:
            adjustment_summary.append({
                'dataset': dataset,
                'min_adjustment': min(adjustments),
                'max_adjustment': max(adjustments),
                'mean_adjustment': np.mean(adjustments),
                'median_adjustment': np.median(adjustments)
            })
    except FileNotFoundError:
        print(f"Could not find variables file for {dataset}")

df_adjustments = pd.DataFrame(adjustment_summary)
print("\nJeffreys Prior Adjustments by Dataset:")
print("=" * 80)
print(df_adjustments.to_string(index=False))
print("\nGlassdoor has much larger adjustments due to large salary values (std ~30K)!")
print("This makes Glassdoor log probs appear ~10 points lower than other datasets.")

## Summary of the Issue

The **Jeffreys prior adjustment is making Glassdoor log probabilities appear implausibly low**.

### The Problem:
1. For normal distributions, the code calculates: `log_prob = log_prob_under_prior - log(population_std)`
2. For Glassdoor, population_std ≈ 30,000 (salary in dollars), so log(30,000) ≈ 10
3. This subtracts ~10 from every log probability, making them appear much worse

### Example from your data:
- **Mean**: 153,075.19
- **Std**: 5,657.21  
- **Ground Truth**: 150,605.64
- **Z-score**: -0.44 (less than half a standard deviation off - quite good!)
- **Log prob under prior**: -9.65 (reasonable)
- **Jeffreys adjustment**: -10.27
- **Final log prob**: **-19.93** (appears terrible, but is actually good!)

### Why this happens:
The Jeffreys prior adjustment is **scale-dependent**. Glassdoor salaries are in the tens of thousands, while other datasets (like NHANES with proportions or smaller numbers) have much smaller scales, leading to smaller adjustments.

### The Fix:
Either:
1. **Remove the Jeffreys adjustment** for comparisons across datasets with different scales
2. **Use a scale-invariant metric** (like z-scores or relative errors)
3. **Apply the adjustment consistently** but be aware it penalizes models in high-variance domains

In [ ]:
# Let's compute the corrected log probs (without Jeffreys adjustment)
results = results_sets[0].copy()

# Add a column for log prob without Jeffreys adjustment
results['log_prob_no_jeffreys'] = np.nan

for idx, row in results.iterrows():
    if row['ground_truth_distribution_type'] == 'normal':
        var_info = vars_file.get(row['variable'], {})
        pop_std = var_info.get('std', None)
        if pop_std is not None:
            jeffreys_adj = np.log(pop_std)
            # Add back the Jeffreys adjustment to get the original log prob
            results.loc[idx, 'log_prob_no_jeffreys'] = row['ground_truth_log_prob'] + jeffreys_adj

# Compare a few examples
print("\nComparison of log probs with and without Jeffreys adjustment:")
print("=" * 80)
sample_results = results[results['ground_truth_distribution_type'] == 'normal'].head(10)

comparison_df = sample_results[['variable', 'mean', 'std', 'ground_truth', 
                                'ground_truth_log_prob', 'log_prob_no_jeffreys']].copy()
comparison_df['difference'] = comparison_df['log_prob_no_jeffreys'] - comparison_df['ground_truth_log_prob']

print(comparison_df.to_string(index=False))

print(f"\n\nAverage Jeffreys adjustment for Glassdoor: {comparison_df['difference'].mean():.2f}")
print("\nThe 'ground_truth_log_prob' column is systematically ~10 points lower than it should be!")